# Validator behaviour, unsupervised: types or a continuum?

| Field | Value |
| --- | --- |
| Question | Do validator nodes fall into distinct behavioural types, or do they spread along a continuum? |
| Unit | One row per node: its whole observed history of staking periods, summarised in 21 behaviour columns |
| Data | The staking-period log, the seamed validator and delegator stakes, the daily monetary ledger, Binance AVAX/USDT and the 3-month T-bill rate, each at the run named in `behaviour.RUNS` |
| Cut | Everything is read as known before 2025-11-19 16:00 UTC, the start of the validation lockbox |
| Library | `avalanche_research.behaviour`: `events`, `features`, `lifetimes`, `unsupervised`, `views` |
| Status | v0, unreviewed |

Restart the kernel and run all. Each cell calls the library and shows its result. Nothing here fits a model that a later cell depends on without saying so.

In [1]:
import warnings

import hvplot.pandas  # noqa: F401
import pandas as pd
import panel as pn
from IPython.display import display

from avalanche_research import behaviour
from avalanche_research.behaviour import events as ev, features as ft, lifetimes as lt, unsupervised as un, views

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*generic' unit for NumPy timedelta", category=DeprecationWarning)
pn.extension("tabulator")
pd.set_option("display.max_columns", 40)

inputs = behaviour.load()
events = ev.decision_events(inputs.periods)
features = ft.build(events, inputs)
life = lt.build(events, features)
life["named_agent"] = inputs.nodes.named_agent.reindex(life.index)
life["registered_actor"] = inputs.nodes.registered_actor.reindex(life.index)
pd.Series(inputs.receipt() | {"decision_events": len(events), "nodes": len(life)})

observed_through                                                    2025-11-19 16:00:00
runs                                  {'binance.avax-usdt-daily': 'binance-avax-dail...
validator_periods                                                                 52941
validator_periods_with_delegations                                                26242
daily_rows                                                                         1896
decision_events                                                                   52129
nodes                                                                              7291
dtype: object

## 1. The observed principal flow

The empirical counterpart of the predecessor's simulated principal-flow Sankey. In one year, validator stake whose period ended either started again on the same node within a week ("renewed in place") or was out for longer. Renewals add or withdraw some stake. Fresh principal is stake that started without being a renewal: a new node's first period, or a return after a gap. Every middle node balances. Choose the year: each ends a week before the lockbox or a whole number of years earlier, so every 7-day label in it is known.

In [2]:
flow = views.PrincipalFlow(events=events, periods=inputs.periods, name="Principal flow")
display(flow.flows()[["source", "target", "million_avax"]])
views.panel(flow, ["window"])

,source,target,million_avax
0,Ended stake,Renewed in place,687.5
1,Ended stake,Out for more than a week,305.8
2,Added at renewal,Renewed in place,28.1
3,Renewed in place,Withdrawn at renewal,40.8
4,Renewed in place,Gross starts,674.9
5,First period of a new node,Fresh principal,72.8
6,Returned within a year,Fresh principal,196.3
7,Returned after more than a year,Fresh principal,8.8
8,Fresh principal,Gross starts,277.9


Row(sizing_mode='stretch_width')
    [0] Param(PrincipalFlow, name='Principal flow', parameters=['window'], show_name=False, widgets={'window': {'sizing_mode':...}, width=260)
    [1] ParamMethod(method, _pane=HoloViews, defer_load=False)

What nodes did at the end of each staking period, by year. `renew` is a next period on the same node within 7 days, `return` within 365, `leave` none within 365, and `censored` a period too recent to tell. This is the empirical counterpart of the predecessor's simulated principal-flow Sankey, by year of the period's end, in AVAX of the ending stake.

In [3]:
flow = (events.assign(year=events.end_time.dt.year.astype(str))
        .groupby(["year", "outcome"], observed=True).stake_avax.sum().div(1e6).rename("stake_m_avax").reset_index())
flow_table = flow.pivot(index="year", columns="outcome", values="stake_m_avax").fillna(0).round(1)
display(ev.summary(events).round(3))
flow.hvplot.bar(x="year", y="stake_m_avax", by="outcome", stacked=True, height=360, responsive=True,
                ylabel="Ending stake, million AVAX", xlabel="", title="What happened to stake at the end of each validator period")

,events,stake_avax,share_of_events,share_of_stake
outcome,,,,
renew,41421,3.309803e+09,0.795,0.717
return,4109,8.578825e+08,0.079,0.186
leave,4348,2.947190e+08,0.083,0.064
overlap,1,9.999998e+03,0.000,0.000
censored,2250,1.547492e+08,0.043,0.034


:Bars   [year,outcome]   (stake_m_avax)

## 2. The lifetime matrix

Each column is mapped to a normal shape by rank, after median imputation, so that heavy tails such as stake do not dominate distances. The principal components show how many directions the behaviour actually varies in.

In [4]:
x = un.matrix(life, lt.BEHAVIOUR)
embedding = un.embed(x, components=6)
display(embedding.explained.round(3).rename("explained variance").to_frame())
embedding.loadings[["PC1", "PC2", "PC3"]].round(2)

,explained variance
PC1,0.315
PC2,0.186
PC3,0.103
PC4,0.071
PC5,0.064
PC6,0.048


,PC1,PC2,PC3
periods_log,0.26,0.25,-0.25
tenure_days_log,0.11,-0.01,-0.09
active_share,-0.25,-0.24,0.24
median_duration_days,0.08,-0.15,-0.03
duration_cv,0.28,0.18,-0.27
short_share,-0.31,0.59,0.31
long_share,0.13,-0.29,-0.00
first_stake_log10,0.24,-0.16,0.33
max_stake_log10,0.31,0.03,0.39
stake_growth_log,0.02,0.04,0.03


## 3. How many groups, against a sample with none

The Bayesian information criterion of a Gaussian mixture falls with every added group. That alone says nothing, because rank-mapped columns with many tied values reward extra components. The sharper test compares the k-means silhouette of the real data with the same score on null samples: data with the same marginals and correlations but a single Gaussian blob, so no groups by construction. A grouping is evidence of types only where it beats every null sample.

In [5]:
display(un.gmm_scan(x, range(1, 9)).round(0))
un.against_null(x, "kmeans", range(2, 9), replicates=5).round(3)

,k,bic,aic,delta_bic
0,1,570891.0,569153.0,621985.0
1,2,335062.0,331580.0,386157.0
2,3,183762.0,178536.0,234857.0
3,4,133668.0,126698.0,184763.0
4,5,92835.0,84121.0,143930.0
5,6,9389.0,-1070.0,60484.0
6,7,-16800.0,-29003.0,34295.0
7,8,-51095.0,-65042.0,0.0


,k,silhouette,null_mean,null_max,beats_null
0,2,0.231,0.158,0.164,True
1,3,0.234,0.141,0.144,True
2,4,0.233,0.130,0.134,True
3,5,0.264,0.125,0.129,True
4,6,0.257,0.112,0.121,True
5,7,0.257,0.101,0.102,True
6,8,0.258,0.100,0.102,True


## 4. Stability of candidate groupings

A grouping that changes when 20% of the nodes are left out is not a type. Each configuration is refitted on ten random 80% subsamples and compared with its full-data labels by the adjusted Rand index: 1 is the same grouping every time.

In [6]:
configurations = {
    "kmeans 3": {"method": "kmeans", "k": 3}, "kmeans 4": {"method": "kmeans", "k": 4},
    "kmeans 5": {"method": "kmeans", "k": 5}, "kmeans 6": {"method": "kmeans", "k": 6},
    "gmm 4": {"method": "gmm", "k": 4}, "ward 4": {"method": "ward", "k": 4},
    "hdbscan": {"method": "hdbscan", "min_cluster_size": 75},
}
comparison = un.compare(x, configurations, rounds=10)
comparison.round(3)

,groups,largest_group_share,silhouette,noise_share,stability_mean,stability_min
configuration,,,,,,
kmeans 3,3,0.465,0.234,0.000,0.992,0.978
kmeans 4,4,0.345,0.233,0.000,0.841,0.574
kmeans 5,5,0.335,0.264,0.000,0.997,0.995
kmeans 6,6,0.334,0.257,0.000,0.876,0.695
gmm 4,4,0.295,0.203,0.000,0.785,0.413
ward 4,4,0.358,0.229,0.000,0.565,0.545
hdbscan,34,0.241,0.406,0.241,0.753,0.688


## 5. The map

Nodes on two principal components. Choose a grouping, or colour by any behaviour column to see whether the groups are regions of a continuum.

In [7]:
groupings = {name: un.fit_labels(x, **config) for name, config in configurations.items()}
lifetime_map = views.LifetimeMap(coordinates=embedding.coordinates,
                                 lifetimes=life.drop(columns=["named_agent", "registered_actor"]),
                                 groupings=groupings, name="Lifetime map")
views.panel(lifetime_map, ["grouping", "colour", "x", "y"])

Row(sizing_mode='stretch_width')
    [0] Param(LifetimeMap, name='Lifetime map', parameters=['grouping', 'colour', ...], show_name=False, widgets={'grouping': {'sizing_mode...}, width=260)
    [1] ParamMethod(method, _pane=HoloViews, defer_load=False)

## 6. What the groups are, in plain units

Medians by group of the most stable configuration with more than two groups, with each group's share of stake-days and the named operators it holds. Names come from the attribution file with their evidence, and are used here only to read the groups.

In [8]:
stable = comparison.loc[comparison.groups.gt(2)].stability_mean.idxmax()
labels = groupings[stable]
shown = ["periods_log", "tenure_days_log", "median_duration_days", "short_share", "long_share", "max_stake_log10",
         "stake_growth_log", "mean_fee_rate", "renew_share", "return_share", "failed_share", "delegation_ratio_mean",
         "largest_delegator_share", "operator_nodes_log", "entry_year", "departed"]
profile = un.profile(life, labels, shown)
named = pd.crosstab(labels, life.named_agent.fillna("unnamed"))
print(f"Most stable grouping with more than two groups: {stable}")
display(profile.round(2))
named

Most stable grouping with more than two groups: kmeans 5


,nodes,stake_days_share,periods_log,tenure_days_log,median_duration_days,short_share,long_share,max_stake_log10,stake_growth_log,mean_fee_rate,renew_share,return_share,failed_share,delegation_ratio_mean,largest_delegator_share,operator_nodes_log,entry_year,departed
group,,,,,,,,,,,,,,,,,,
0,994,0.03,0.69,4.50,88.70,0.00,0.0,3.30,0.00,0.02,0.0,0.0,0.0,0.10,0.0,0.69,2022.0,1.0
1,1305,0.00,1.79,5.54,46.01,0.00,0.0,3.30,0.00,0.03,0.857143,0.0,0.0,0.01,0.0,3.18,2023.0,1.0
2,1541,0.00,0.69,2.77,15.00,1.00,0.0,3.30,0.00,0.02,0.0,0.0,0.0,0.00,0.0,7.02,2024.0,0.0
3,2442,0.95,2.08,6.20,41.52,0.03,0.0,3.72,0.01,0.02,0.777778,0.0,0.0,0.49,0.0,0.69,2021.0,1.0
4,1009,0.01,2.20,4.98,15.00,1.00,0.0,3.31,0.00,0.02,0.916667,0.0,0.0,0.05,0.0,6.89,2024.0,0.0


named_agent,BENQI,GoGoPool (Hypha),mainnet genesis allocation holder,unnamed,unnamed operator of 24 nodes
kmeans5,,,,,
0,35,1,0,957,1
1,97,60,237,888,23
2,77,1036,82,346,0
3,8,19,1,2412,2
4,4,784,0,221,0


## 7. The same without the registered actors

Two of the groups above are mostly GoGoPool minipools, which share one reward address and a 2-week cycle. Leaving out the registered actors (GoGoPool and the genesis allocation, named on registry evidence) shows whether the structure survives without them.

In [9]:
free = life.registered_actor.isna()
x_free = un.matrix(life[free], lt.BEHAVIOUR)
comparison_free = un.compare(x_free, configurations, rounds=10)
stable_free = comparison_free.loc[comparison_free.groups.gt(2)].stability_mean.idxmax()
labels_free = un.fit_labels(x_free, **configurations[stable_free])
print(f"Most stable grouping without the registered actors: {stable_free}")
display(un.against_null(x_free, "kmeans", range(2, 9), replicates=5).round(3))
display(comparison_free.round(3))
un.profile(life[free], labels_free, shown).round(2)

Most stable grouping without the registered actors: kmeans 3


,k,silhouette,null_mean,null_max,beats_null
0,2,0.227,0.149,0.151,True
1,3,0.217,0.122,0.131,True
2,4,0.227,0.105,0.109,True
3,5,0.205,0.096,0.099,True
4,6,0.207,0.091,0.097,True
5,7,0.205,0.090,0.091,True
6,8,0.200,0.087,0.089,True


,groups,largest_group_share,silhouette,noise_share,stability_mean,stability_min
configuration,,,,,,
kmeans 3,3,0.518,0.217,0.000,0.999,0.998
kmeans 4,4,0.517,0.227,0.000,0.926,0.576
kmeans 5,5,0.292,0.205,0.000,0.981,0.941
kmeans 6,6,0.272,0.207,0.000,0.870,0.865
gmm 4,4,0.305,0.165,0.000,0.650,0.513
ward 4,4,0.536,0.226,0.000,0.957,0.905
hdbscan,24,0.333,0.311,0.333,0.272,0.089


,nodes,stake_days_share,periods_log,tenure_days_log,median_duration_days,short_share,long_share,max_stake_log10,stake_growth_log,mean_fee_rate,renew_share,return_share,failed_share,delegation_ratio_mean,largest_delegator_share,operator_nodes_log,entry_year,departed
group,,,,,,,,,,,,,,,,,,
0,2627,0.94,1.95,6.10,34.17,0.08,0.0,3.68,0.01,0.02,0.772727,0.0,0.0,0.41,0.0,0.69,2021.0,1.0
1,1378,0.03,0.69,3.47,31.01,0.00,0.0,3.30,0.00,0.02,0.0,0.0,0.0,0.01,0.0,0.69,2022.0,1.0
2,1066,0.03,1.39,5.63,61.51,0.00,0.0,3.30,0.00,0.02,0.666667,0.0,0.0,0.02,0.0,1.95,2023.0,1.0


## 8. The most unusual nodes

An isolation forest scores how easily each node is separated from the rest. The most unusual nodes are candidates for named operators, data faults, or behaviour the groups do not describe.

In [10]:
anomaly = un.outliers(x)
life.assign(anomaly=anomaly).sort_values("anomaly", ascending=False).head(15)[
    ["anomaly", "named_agent", "periods_log", "median_duration_days", "max_stake_log10", "mean_fee_rate",
     "renew_share", "delegation_ratio_mean", "operator_nodes_log", "entry_year"]].round(2)

,anomaly,named_agent,periods_log,median_duration_days,max_stake_log10,mean_fee_rate,renew_share,delegation_ratio_mean,operator_nodes_log,entry_year
node_id,,,,,,,,,,
NodeID-4fK92LkTyEUzPoDW44Bo9b5YvL5kJ7369,0.63,None,1.10,354.42,3.30,0.20,0.0,0.00,0.69,2023
NodeID-Gfeb3fWPCSpA4g9edG68NS5aJcYpXn6T3,0.63,None,1.10,354.43,3.30,0.20,0.0,0.00,0.69,2023
NodeID-FjNGj9nt6NhxUqifWMENfLk5q7ErinWWM,0.63,None,1.10,14.00,4.97,0.98,0.0,0.00,0.69,2025
NodeID-D3onstuMsGRctDjbksXU6BV3rrCbWHkB9,0.63,None,1.61,365.00,5.53,0.03,0.75,23.05,0.69,2020
NodeID-BBBH9ALTvkTyVReh54fAhagFsX7eWGqJ3,0.62,None,0.69,14.00,3.30,0.02,0.0,0.00,0.69,2020
NodeID-FVMb2XgKVaeonehFbPMe714kMJ7DRxhpt,0.62,None,1.10,153.59,4.60,0.02,0.0,16.65,5.54,2023
NodeID-NQ7btBfH5MdLFwZwAR2F6UHRgHXjfNPYV,0.62,None,1.10,14.00,3.64,0.04,0.5,0.00,0.69,2020
NodeID-N1ZQvBrWWmu5TryPdQphR3mcxNouZoaL7,0.62,None,1.10,365.00,4.11,0.04,0.5,38.28,0.69,2021
NodeID-ExjkLyBwH98GZ4mgLcsF38Xt63x9NAvaX,0.62,None,0.69,182.44,3.30,0.03,0.0,32.73,1.10,2024


## Findings

Written from the run above. Codes are `UF1` onward.

| Code | Finding |
| --- | --- |
| `UF0` | In the year to 2025-11-12, 993 million AVAX of validator stake ended its period. 69% started again on the same node within a week, net of 28 million added and 41 million withdrawn at renewal, and 31% was out for longer. Gross starts were 953 million, of which 278 million was fresh principal: 196 million returning within a year, 73 million from new nodes, 9 million returning after longer. The renewed share of ended stake fell from 84% (2020 to 2021) to 57% (2023 to 2024) and recovered to 69%. Returns after a gap peaked at 370 million AVAX in 2023 to 2024, while new nodes' first stake fell from 151 to 51 million |
| `UF1` | Of 52,129 validator periods that ended before the lockbox, 79.5% were followed by a next period on the same node within 7 days, 7.9% by one later in the year, and 8.3% by none, with 4.3% too recent to tell. By ending stake the split is 71.7%, 18.6% and 6.4%: large validators are the ones that come back late. In 2024, 320 of 939 million AVAX of ending stake returned after more than a week |
| `UF2` | The node lifetimes have structure: for every number of groups from 2 to 8, the k-means silhouette (0.23 to 0.26) beats every null sample that keeps each column's exact marginal and the columns' rank correlations (at most 0.16). The silhouettes are low in absolute terms, so the groups are dense regions that overlap, not separated types |
| `UF3` | On every node, five k-means groups are stable under resampling (mean adjusted Rand 0.997). Two of them are mostly GoGoPool minipools (1,820 of the 1,900 GoGoPool nodes with an ended period), defined by 2-week periods and a reward address shared by about a thousand nodes. One group of 2,442 long-lived nodes holds 95% of stake-days |
| `UF4` | Without the registered actors (5,071 nodes), three k-means groups are the most stable (0.999), and five groups (0.981, never below 0.94) and Ward's four (0.957) also hold. HDBSCAN is unstable (0.27) with a third of nodes as noise. The three are 2,627 long-lived nodes that attract delegation (median delegation ratio 0.41 of own stake) and hold 94% of stake-days, 1,378 nodes with a single period, and 1,066 nodes with fewer and longer periods (median 62 days) whose reward addresses are shared by a handful of nodes |
| `UF5` | So the stable picture is a dominant persistent population and two transient ones, with finer structure that is stable at five groups but overlapping. Operators, rather than nodes, are the next unit to test, once identity resolution gives them |
| `UF6` | The most unusual nodes include several with identical histories: one or two periods of about 354 days at 2,000 AVAX and a 20% fee. They are candidates for one operator's batch, to check against the identity work |